## Patient Journey - Diagnosis
#### Data source used
* PHCDW.PHCDW_DSAA.IQV_LAAD_DX_VW
* PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_DIAGNOSIS
* PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PRODUCT 

#### Steps:
1. Query CKD diagnosis claims data after 2016-01-01
2. Get date level diagnosis stage records (get the daily maximium if multiple records for one date, forward fill for missing between two diagnosis) for each patient
3. Save table to databricks

#### Notes:
1. Time frame: SVC_DT >= '2017-01-01' 
2. CKD stage:     
    * when CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 1'  then 1
    * when CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 2 (MILD)'  then 2
    * when CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 3 (MODERATE)'  then 3
    * when CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 4 (SEVERE)'  then 4
    * when CATEGORY like '%STAGE 5%'  then 5
3. Check the the end_date of the data, now we set end_date_limit = Max(SVC_DT)

## 1. Setup Snowflake connection and load data

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
%run "../../00_cohort_config"

In [0]:
import pandas as pd
import pyspark.sql.functions as F
# Source: the 3.0 "Pat Journey Dx" query (00_Write_Input_Files), run directly from Snowflake.
# The pre-built query_output/pat_journey_dx table was removed in the 2026-10-26 refresh.
print(f"Dx query window: {analysis_start_month} -> {current_month}")
dx_query = f"""
 SELECT DISTINCT
   NPI_NUMBER,
   PROVIDER_ID,
   PATIENT_ID,
   SVC_DT,
   CATEGORY as DX_STAGE_TEXT,
   CASE
      when CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 1'  then 1
      when CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 2 (MILD)'  then 2
      when CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 3 (MODERATE)'  then 3
      when CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 4 (SEVERE)'  then 4
      when CATEGORY like '%STAGE 5%'  then 5
      else 0
      end as DX_STAGE
 FROM
 (
  SELECT  DISTINCT
                   NPI_NUMBER,
                   PRIMARY_SPECIALTY_DESC,
                   PATIENT_ID,SERVICE_DATE AS SVC_DT,
                   CD.DIAGNOSIS_DESCRIPTION as Category,
                   COALESCE(PROVIDER_REFERRING_ID, PROVIDER_RENDERING_ID) as PROVIDER_ID
  FROM (Select
C.NPI_NUMBER,
A.PATIENT_ID,
A.SERVICE_DATE,
A.PROVIDER_RENDERING_ID,
A.PROVIDER_REFERRING_ID,
A.DIAGNOSIS_CODE,
B.PATIENT_BIRTH_YEAR,
C.PRIMARY_SPECIALTY_CODE,
C.PRIMARY_SPECIALTY_DESC
from  (select PATIENT_ID, CLAIM_ID, DIAGNOSIS_CODE, SERVICE_DATE, PROVIDER_RENDERING_ID, PROVIDER_REFERRING_ID  FROM
PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_DX) A
left outer join
(select distinct PATIENT_ID, PATIENT_BIRTH_YEAR, PATIENT_GENDER from PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PTNT_DMGRPHC) B
on A.PATIENT_ID = B.PATIENT_ID
left outer join
PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR C
on COALESCE(A.PROVIDER_REFERRING_ID,A.PROVIDER_RENDERING_ID) = C.PROVIDER_ID
) DX
  JOIN (
        SELECT DISTINCT DIAGNOSIS_CODE, DIAGNOSIS_DESCRIPTION
        FROM PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_DIAGNOSIS
        WHERE DIAGNOSIS_CODE IN (
          '249.4', '249.40', '249.41', '250.4', '250.40', '250.42', '285.21', '403', '403.0', '403.00', '403.1', '403.10', '403.9',
          '403.90', '404', '404.0', '404.00', '404.01', '404.1', '404.10', '404.11', '404.9', '404.90', '404.91', '585', '585.1',
          '585.2', '585.3', '585.4', '585.9', '588.81', 'D63.1', 'E08.2', 'E08.21','E08.22', 'E08.29', 'E09.2', 'E09.21', 'E09.22',
          'E09.29', 'E11.2', 'E11.21', 'E11.22', 'E11.29', 'E13.2', 'E13.21', 'E13.22', 'E13.29', 'I12', 'I12.9', 'I13', 'I13.0',
          'I13.1', 'I13.10', 'N18', 'N18.1', 'N18.2', 'N18.3', 'N18.4', 'N18.9', 'N25.81', '503.60' , '503.65', 'S20.65', '0TY0.0Z0',
          '0TY0.0Z1', '0TY0.0Z2', '0TY1.0Z0', '0TY1.0Z1', '0TY1.0Z2', 'Z94.0', 'V42.0', '556.9', '909.18', '909.19', '909.20', '909.21',
          '909.22', '909.23', '909.24', '909.25', '909.35', '909.37', '909.45', '909.47', '909.51', '909.52', '909.53', '909.54',
          '909.55', '909.56', '909.57', '909.58', '909.59', '909.60', '909.61', '909.62', '909.63', '909.64', '909.65', '909.66',
          '909.67', '909.68', '909.69', '909.70', '909.99', '995.12', '995.59', 'G02.57', 'G03.08', 'G03.09', 'G03.10', 'G03.11',
          'G03.12', 'G03.13', 'G03.14', 'G03.15', 'G03.16', 'G03.17', 'G03.18', 'G03.19', 'G03.20', 'G03.21', 'G03.22', 'G03.23',
          'G03.24', 'G03.25', 'G03.26', 'G03.27', 'G90.13', 'G90.14', 'S93.35', 'S93.39','3E1.M39Z', '5A1.D00Z', '5A1.D60Z',
          '5A1.D70Z', '5A1.D80Z', '5A1.D90Z', 'Z99.2', '399.5', '549.8', '800', '801', '802', '803', '804', '809', '820', '821',
          '829', '830', '831', '839', '840', '841', '849', '850', '851', '859', '880', '881', '889')
        ) CD
  ON DX.DIAGNOSIS_CODE=CD.DIAGNOSIS_CODE
   WHERE (EXTRACT(YEAR FROM SVC_DT)*100 + EXTRACT(MONTH FROM SVC_DT)) >= '{analysis_start_month}'
    AND (EXTRACT(YEAR FROM SVC_DT)*100 + EXTRACT(MONTH FROM SVC_DT)) <= '{current_month}' AND COALESCE(PROVIDER_REFERRING_ID,PROVIDER_RENDERING_ID) IS NOT NULL
  )
"""

df_dx = Get_Data_Snowflakes(dx_query)

# HCP universe (same as 3.0 00_Write_Input_Files)
_hcp_universe = spark.read.format("delta").load(hcp_universe_path).select("PROVIDER_ID").distinct()
df_dx = df_dx.join(_hcp_universe, how="inner", on="PROVIDER_ID")

# Filter to data >= DATA_START_DATE (from config)
df_dx = df_dx.filter(F.col("SVC_DT") >= DATA_START_DATE)

# Validation filter: 10 test HCPs (from config)
df_dx, test_patients = apply_hcp_filter(df_dx)

pd.DataFrame(df_dx.columns).display()

QC

In [0]:
# df_dx_dev = spark.read.format("parquet").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/dev_folder/query_output/pat_journey_dx.parquet")
# pd.DataFrame(df_dx_dev.columns).display()

In [0]:
# overlap_patients = df_dx.select("PATIENT_ID").distinct().join(df_dx_dev.select("PATIENT_ID").distinct(), on = "PATIENT_ID", how = "inner")
# overlap_patients.show(5)

In [0]:
# df1 = df_dx.select("PATIENT_ID", "SVC_DT", "DX_STAGE")
# df1 = df1.withColumnRenamed("DX_STAGE", "DX_STAGE_OPS")
# df2 = df_dx_dev.select("PATIENT_ID", "SVC_DT", "DX_STAGE")
# df2 = df2.withColumnRenamed("DX_STAGE", "DX_STAGE_DEV")
# df_comp = df1.join(df2, on = ['PATIENT_ID', 'SVC_DT'], how = "left")
# # print(df_comp.filter(F.col('DX_STAGE_OPS') != F.col('DX_STAGE_DEV')).select('PATIENT_ID').distinct().count())
# df_comp.filter(F.col('DX_STAGE_OPS') != F.col('DX_STAGE_DEV')).display()

## 2. Data cleanup and processing

In [0]:
from datetime import datetime
from pyspark.sql.types import DateType, StructType, StructField, StringType
from pyspark.sql import functions as F, Window
import sys

# Function to generate cohort-month reference dates (1st of each month)
def generate_month_range(start, end):
    current = datetime.strptime(start, "%Y-%m-%d")
    end = datetime.strptime(end, "%Y-%m-%d")
    months = []
    while current <= end:
        months.append(current.strftime("%Y-%m-01"))
        if current.month == 12:
            current = current.replace(year=current.year + 1, month=1, day=1)
        else:
            current = current.replace(month=current.month + 1, day=1)
    return months

# Step 1: Get min and max SVC_DT, convert to cohort-months using offset
min_dt, max_dt = df_dx.agg(F.min("SVC_DT"), F.max("SVC_DT")).first()
if OFFSET_DAY == 0:
    min_month_str = min_dt.strftime("%Y-%m-01")
    max_month_str = max_dt.strftime("%Y-%m-01")
else:
    from datetime import timedelta
    min_shifted = min_dt - timedelta(days=OFFSET_DAY - 1)
    max_shifted = max_dt - timedelta(days=OFFSET_DAY - 1)
    min_month_str = min_shifted.strftime("%Y-%m-01")
    max_month_str = max_shifted.strftime("%Y-%m-01")

# Step 2: Create DataFrame of all possible cohort-months for all PATIENT_IDs
all_months = generate_month_range(min_month_str, max_month_str)
months_schema = StructType([StructField("COHORT_MONTH", StringType(), True)])
months_df = spark.createDataFrame([(m,) for m in all_months], schema=months_schema)

patient_ids = df_dx.select("PATIENT_ID").distinct()
patient_months_df = patient_ids.crossJoin(months_df)

# Step 3: Map SVC_DT to cohort-month and group
df_dx_monthly = (
    df_dx
    .withColumn("COHORT_MONTH", F.date_format(assign_cohort_month(F.col("SVC_DT")), "yyyy-MM-01"))
    .groupBy("PATIENT_ID", "COHORT_MONTH")
    .agg(F.collect_set("DX_STAGE").alias("DX_STAGE"))
)

# Step 4: Join complete patient-cohort-month grid with aggregated data
df_dx_monthly = (
    patient_months_df
    .join(df_dx_monthly, on=["PATIENT_ID", "COHORT_MONTH"], how="left")
)

# Step 5: Compute max stage per cohort-month
df_dx_monthly = (
    df_dx_monthly
    .withColumn("DX_STAGE_MAX", F.array_max(F.col("DX_STAGE")))
    .withColumn("DX_STAGE_MAX", F.when(F.col("DX_STAGE_MAX") == 0, None).otherwise(F.col("DX_STAGE_MAX")))
)

# Step 6: Define forward fill window
w_forward = (
    Window.partitionBy("PATIENT_ID")
    .orderBy("COHORT_MONTH")
    .rowsBetween(-sys.maxsize, 0)
)

# Step 7: Forward fill DX_STAGE_MAX
df_dx_monthly_filled = (
    df_dx_monthly
    .withColumn("DX_STAGE_UPDATED", F.last("DX_STAGE_MAX", ignorenulls=True).over(w_forward))
)

# Combine and fill nulls with 0
df_dx_combine = (
    df_dx_monthly
    .join(df_dx_monthly_filled.select("PATIENT_ID", "COHORT_MONTH", "DX_STAGE_UPDATED"), 
          on=["PATIENT_ID", "COHORT_MONTH"], how="left")
    .na.fill(value=0, subset=["DX_STAGE_UPDATED"])
)

# Join cohort labels (C1-C22); rows before C1 get null label (lookback)
df_dx_combine = df_dx_combine.join(
    df_cohort_ref.select(
        F.date_format("cohort_ref_month", "yyyy-MM-01").alias("_ref_month_str"),
        "cohort_label"
    ),
    df_dx_combine.COHORT_MONTH == F.col("_ref_month_str"),
    "left"
).drop("_ref_month_str")

df_dx_combine.display()

In [0]:
df_dx_combine = df_dx_combine.withColumnRenamed("COHORT_MONTH", "DATE")


df_dx_combine = df_dx_combine.drop("DX_STAGE_MAX")


In [0]:
print('Shape:',df_dx_combine.count(), len(df_dx_combine.columns))
print('# Unique HCPs:',df_dx_combine.select("PATIENT_ID").distinct().count())
print('Min Date:',df_dx_combine.agg({"DATE": "min"}).collect()[0][0])
print('Max Date:', df_dx_combine.agg({"DATE": "max"}).collect()[0][0])

In [0]:
df_dx_combine.printSchema()

In [0]:
df_dx_combine = df_dx_combine.withColumn("DATE", F.to_date(F.col("DATE"), "yyyy-MM-dd"))

In [0]:
# COMPARISON: offset=0 validation vs production
base = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output"
prod_dx = spark.read.format("delta").load(f"{base}/df_dx_combine")
val_pats = [r.PATIENT_ID for r in df_dx_combine.select("PATIENT_ID").distinct().collect()]
prod_dx_f = prod_dx.filter(F.col("PATIENT_ID").isin(val_pats))

val_c = df_dx_combine.count()
prod_c = prod_dx_f.count()
print(f"Validation (offset=0): {val_c:,}")
print(f"Production (same pts): {prod_c:,}")
print(f"Difference:            {val_c - prod_c:+,}")

val_k = df_dx_combine.select("PATIENT_ID", "DATE")
prod_k = prod_dx_f.select("PATIENT_ID", F.col("DATE").cast("date").alias("DATE"))
only_val = val_k.subtract(prod_k).count()
only_prod = prod_k.subtract(val_k).count()
print(f"\nRows only in validation: {only_val}")
print(f"Rows only in production: {only_prod}")
prod_only_before = prod_k.subtract(val_k).filter(F.col("DATE") < DATA_START_DATE).count()
print(f"  Prod-only before {DATA_START_DATE}: {prod_only_before}")
print(f"  Prod-only on/after {DATA_START_DATE}: {only_prod - prod_only_before}")

# Stage diffs on overlapping rows
common = df_dx_combine.alias("v").join(
    prod_dx_f.withColumn("DATE", F.col("DATE").cast("date")).alias("p"),
    (F.col("v.PATIENT_ID") == F.col("p.PATIENT_ID")) & (F.col("v.DATE") == F.col("p.DATE")), "inner"
)
diffs = common.filter(
    (F.col("v.DX_STAGE_UPDATED") != F.col("p.DX_STAGE_UPDATED")) |
    (F.col("v.DX_STAGE_UPDATED").isNull() != F.col("p.DX_STAGE_UPDATED").isNull())
).count()
print(f"\nOverlapping rows: {common.count():,}")
print(f"Stage diffs: {diffs}")

In [0]:
# VALIDATION: write to a separate path so we don't overwrite production data
df_dx_combine.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_dx_journey_validation")